# XGBoost hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
# Create subsets of the data for different training sizes - chronological order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

## Classification

[Parameters](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html)

### 1k

In [8]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [9]:
import optuna
import numpy as np
import warnings
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 10.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree 
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 0.8), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2, 10), # Class balancing weight for imbalanced target
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "objective": "binary:logistic", # Binary classification objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
        
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0)
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBClassifier(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "scale_pos_weight",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_1k_parallel.html")


[I 2026-04-22 23:27:38,003] A new study created in memory with name: no-name-638ee5c6-177f-41e2-9649-9d42c6215288
[I 2026-04-22 23:27:38,598] Trial 0 finished with value: 0.6235971595454354 and parameters: {'booster': 'gbtree', 'learning_rate': 0.060916162117028466, 'min_split_loss': 8.393759192056747, 'max_depth': 2, 'min_child_weight': 0.005750186383248772, 'max_delta_step': 2.9157810676314435, 'colsample_bytree': 0.4142301971387032, 'colsample_bylevel': 0.5752583592551246, 'colsample_bynode': 0.6810800804265573, 'reg_lambda': 5.911414529989623, 'reg_alpha': 0.001451010765676258, 'scale_pos_weight': 7.5134196444262225, 'grow_policy': 'depthwise', 'max_leaves': 25, 'max_bin': 231, 'max_cat_to_onehot': 7, 'max_cat_threshold': 28, 'n_estimators': 124, 'sampling_method': 'uniform', 'subsample': 0.5665293313684052}. Best is trial 0 with value: 0.6235971595454354.
[I 2026-04-22 23:27:38,773] Trial 3 finished with value: 0.656485458985459 and parameters: {'booster': 'gbtree', 'learning_rate


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-22 23:29:18,981] Trial 278 finished with value: 0.6659561798354902 and parameters: {'booster': 'gbtree', 'learning_rate': 0.009129651465639259, 'min_split_loss': 1.4830383794157913, 'max_depth': 3, 'min_child_weight': 0.0011374790544290228, 'max_delta_step': 8.800566402307224, 'colsample_bytree': 0.5973821414025523, 'colsample_bylevel': 0.6244567293453808, 'colsample_bynode': 0.30246029548647385, 'reg_lambda': 0.0037568727361846606, 'reg_alpha': 4.00962678753063, 'scale_pos_weight': 2.2964636702088006, 'grow_policy': 'lossguide', 'max_leaves': 29, 'max_bin': 228, 'max_cat_to_onehot': 9, 'max_cat_threshold': 26, 'n_estimators': 963, 'sampling_method': 'uniform', 'subsample': 0.6046158244166475}. Best is trial 177 with value: 0.6732099031236962.
[I 2026-04-22 23:29:19,179] Trial 279 finished with value: 0.6580379295034468 and parameters: {'booster': 'gbtree', 'learning_rate': 0.009395068553465048, 'min_split_loss': 8.841048576529545, 'max_depth': 3, 'min_child_weight': 0.00130


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-22 23:29:19,475] Trial 280 finished with value: 0.6656615510925855 and parameters: {'booster': 'gbtree', 'learning_rate': 0.009266726556969408, 'min_split_loss': 8.437626921572967, 'max_depth': 3, 'min_child_weight': 0.0018458563832359069, 'max_delta_step': 8.647345248793883, 'colsample_bytree': 0.5562641618617797, 'colsample_bylevel': 0.6761440233389999, 'colsample_bynode': 0.6004534385696364, 'reg_lambda': 0.0128083984129407, 'reg_alpha': 3.955036864331389, 'scale_pos_weight': 2.3058080335065996, 'grow_policy': 'lossguide', 'max_leaves': 29, 'max_bin': 219, 'max_cat_to_onehot': 10, 'max_cat_threshold': 28, 'n_estimators': 741, 'sampling_method': 'uniform', 'subsample': 0.5988163705490936}. Best is trial 177 with value: 0.6732099031236962.
[I 2026-04-22 23:29:19,577] Trial 282 finished with value: 0.66557048315669 and parameters: {'booster': 'gbtree', 'learning_rate': 0.009336829851323345, 'min_split_loss': 2.0006116912133685, 'max_depth': 3, 'min_child_weight': 0.001293770


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6732
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.049961658713390054,
    "min_split_loss": 3.1792890444034416,
    "max_depth": 4,
    "min_child_weight": 0.03446885315674068,
    "max_delta_step": 8.818441265852027,
    "colsample_bytree": 0.6612602163953242,
    "colsample_bylevel": 0.4402275230048245,
    "colsample_bynode": 0.44195112565215633,
    "reg_lambda": 3.821278757725177,
    "reg_alpha": 5.339764624690546,
    "scale_pos_weight": 6.723400891593152,
    "grow_policy": "depthwise",
    "max_leaves": 23,
    "max_bin": 247,
    "max_cat_to_onehot": 8,
    "max_cat_threshold": 26,
    "n_estimators": 614,
    "sampling_method": "uniform",
    "subsample": 0.6152248546699406,
}

--- PARAMETER IMPORTANCE ---
  reg_alpha           :

In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_xgboost.best_params.copy()

best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "binary:logistic"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_xgboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)

BEST_PARAMS: {'booster': 'gbtree', 'learning_rate': 0.049961658713390054, 'min_split_loss': 3.1792890444034416, 'max_depth': 4, 'min_child_weight': 0.03446885315674068, 'max_delta_step': 8.818441265852027, 'colsample_bytree': 0.6612602163953242, 'colsample_bylevel': 0.4402275230048245, 'colsample_bynode': 0.44195112565215633, 'reg_lambda': 3.821278757725177, 'reg_alpha': 5.339764624690546, 'scale_pos_weight': 6.723400891593152, 'grow_policy': 'depthwise', 'max_leaves': 23, 'max_bin': 247, 'max_cat_to_onehot': 8, 'max_cat_threshold': 26, 'n_estimators': 614, 'sampling_method': 'uniform', 'subsample': 0.6152248546699406, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'eval_metric': 'auc', 'enable_categorical': True}

Optuna Cross-Val AUC: 0.6732
Holdout Test AUC:     0.6866


### 10k

In [4]:
import optuna
import numpy as np
import warnings
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

## Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 10.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree 
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 0.8), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2, 10), # Class balancing weight for imbalanced target
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "objective": "binary:logistic", # Binary classification objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
        
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0)
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBClassifier(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "scale_pos_weight",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_10k_parallel.html")


[I 2026-04-25 13:01:21,851] A new study created in memory with name: no-name-8b9ada65-73c8-4a1d-82ae-19add5ebf2ae
[I 2026-04-25 13:01:24,721] Trial 6 finished with value: 0.6900484301906724 and parameters: {'booster': 'gbtree', 'learning_rate': 0.03341020384423113, 'min_split_loss': 7.312489426991165, 'max_depth': 2, 'min_child_weight': 0.0016822568601802727, 'max_delta_step': 4.08635306864467, 'colsample_bytree': 0.3943760681933986, 'colsample_bylevel': 0.37048644302659683, 'colsample_bynode': 0.7582898633547546, 'reg_lambda': 0.001621157713725593, 'reg_alpha': 0.334372106804997, 'scale_pos_weight': 4.585413613161524, 'grow_policy': 'lossguide', 'max_leaves': 9, 'max_bin': 216, 'max_cat_to_onehot': 2, 'max_cat_threshold': 28, 'n_estimators': 323, 'sampling_method': 'uniform', 'subsample': 0.9403766638140618}. Best is trial 6 with value: 0.6900484301906724.
[I 2026-04-25 13:01:26,474] Trial 2 finished with value: 0.6776050502346453 and parameters: {'booster': 'gbtree', 'learning_rate':


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 13:08:28,207] Trial 376 finished with value: 0.6985215525830301 and parameters: {'booster': 'gbtree', 'learning_rate': 0.0054184358973992115, 'min_split_loss': 3.954412922531746, 'max_depth': 3, 'min_child_weight': 2.5361187877479776, 'max_delta_step': 3.120732475016469, 'colsample_bytree': 0.3400666866360873, 'colsample_bylevel': 0.3119489800182622, 'colsample_bynode': 0.6618848785748406, 'reg_lambda': 0.5652843736532919, 'reg_alpha': 0.2686705834564865, 'scale_pos_weight': 2.697461229627068, 'grow_policy': 'depthwise', 'max_leaves': 13, 'max_bin': 245, 'max_cat_to_onehot': 4, 'max_cat_threshold': 5, 'n_estimators': 833, 'sampling_method': 'gradient_based', 'subsample': 0.2876069310300584}. Best is trial 277 with value: 0.702101361218537.
[I 2026-04-25 13:08:30,315] Trial 379 finished with value: 0.7002987547402355 and parameters: {'booster': 'gbtree', 'learning_rate': 0.005793418027694442, 'min_split_loss': 4.776556716967011, 'max_depth': 2, 'min_child_weight': 1.522153


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 13:08:31,585] Trial 380 finished with value: 0.7003505246861356 and parameters: {'booster': 'gbtree', 'learning_rate': 0.005767658177776413, 'min_split_loss': 4.983584772870444, 'max_depth': 3, 'min_child_weight': 0.32034141427521357, 'max_delta_step': 3.081866711819175, 'colsample_bytree': 0.3389640263085274, 'colsample_bylevel': 0.30932261955410395, 'colsample_bynode': 0.6847548659341708, 'reg_lambda': 0.30470028252671383, 'reg_alpha': 0.28381450636887084, 'scale_pos_weight': 2.3921090389285156, 'grow_policy': 'depthwise', 'max_leaves': 17, 'max_bin': 249, 'max_cat_to_onehot': 3, 'max_cat_threshold': 7, 'n_estimators': 831, 'sampling_method': 'gradient_based', 'subsample': 0.312598574901212}. Best is trial 277 with value: 0.702101361218537.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 13:08:31,926] Trial 381 finished with value: 0.699926713453961 and parameters: {'booster': 'gbtree', 'learning_rate': 0.005006610369312247, 'min_split_loss': 5.120232873452065, 'max_depth': 3, 'min_child_weight': 1.9824409289823455, 'max_delta_step': 3.1448731965911927, 'colsample_bytree': 0.3015069017834314, 'colsample_bylevel': 0.34802436085639166, 'colsample_bynode': 0.7198065903279282, 'reg_lambda': 0.48826803737586305, 'reg_alpha': 0.19182203593110206, 'scale_pos_weight': 2.7491574873721576, 'grow_policy': 'depthwise', 'max_leaves': 13, 'max_bin': 250, 'max_cat_to_onehot': 3, 'max_cat_threshold': 7, 'n_estimators': 822, 'sampling_method': 'gradient_based', 'subsample': 0.3174082822502198}. Best is trial 277 with value: 0.702101361218537.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 13:08:32,403] Trial 383 finished with value: 0.7010644550946246 and parameters: {'booster': 'gbtree', 'learning_rate': 0.005761561228191891, 'min_split_loss': 4.034959843001035, 'max_depth': 3, 'min_child_weight': 3.234246127528609, 'max_delta_step': 2.7910810677510822, 'colsample_bytree': 0.3399439484409881, 'colsample_bylevel': 0.3095984133486481, 'colsample_bynode': 0.6820889822206211, 'reg_lambda': 0.1887250768203031, 'reg_alpha': 0.30285392364790437, 'scale_pos_weight': 2.3946625169273092, 'grow_policy': 'depthwise', 'max_leaves': 13, 'max_bin': 216, 'max_cat_to_onehot': 4, 'max_cat_threshold': 7, 'n_estimators': 822, 'sampling_method': 'gradient_based', 'subsample': 0.3380029324308754}. Best is trial 277 with value: 0.702101361218537.
[I 2026-04-25 13:08:32,575] Trial 382 finished with value: 0.699332151402237 and parameters: {'booster': 'gbtree', 'learning_rate': 0.00585476810605333, 'min_split_loss': 4.911192986542424, 'max_depth': 3, 'min_child_weight': 3.5951924


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7021
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.005321269756124328,
    "min_split_loss": 1.5564766508977637,
    "max_depth": 3,
    "min_child_weight": 0.002666795494419774,
    "max_delta_step": 3.237106539752366,
    "colsample_bytree": 0.6334129748685073,
    "colsample_bylevel": 0.31232752047853835,
    "colsample_bynode": 0.5663544691691175,
    "reg_lambda": 0.866361380309008,
    "reg_alpha": 0.34097661390173023,
    "scale_pos_weight": 2.237901939773783,
    "grow_policy": "lossguide",
    "max_leaves": 14,
    "max_bin": 203,
    "max_cat_to_onehot": 2,
    "max_cat_threshold": 4,
    "n_estimators": 914,
    "sampling_method": "gradient_based",
    "subsample": 0.21905666352297487,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.7632
  colsample_bylevel   : 0.0844
  max_delta_step  

In [5]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_xgboost.best_params.copy() 

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

if best_params["booster"] == "gbtree":
    # Apply trick only for gbtree
    best_params["n_estimators"] = original_trees * 2
    best_params["learning_rate"] = original_lr / 2
    print(f"\n[Scaling Trick Applied] GBTREE: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")
else:
    # Keep original parameters for DART
    print(f"\n[Scaling Trick Skipped] Booster is DART. Kept original values: Trees {original_trees}, LR {original_lr:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "binary:logistic"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_xgboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] GBTREE: Trees 914 -> 1828, LR 0.0053 -> 0.0027
BEST PARAMS: {'booster': 'gbtree', 'learning_rate': 0.002660634878062164, 'min_split_loss': 1.5564766508977637, 'max_depth': 3, 'min_child_weight': 0.002666795494419774, 'max_delta_step': 3.237106539752366, 'colsample_bytree': 0.6334129748685073, 'colsample_bylevel': 0.31232752047853835, 'colsample_bynode': 0.5663544691691175, 'reg_lambda': 0.866361380309008, 'reg_alpha': 0.34097661390173023, 'scale_pos_weight': 2.237901939773783, 'grow_policy': 'lossguide', 'max_leaves': 14, 'max_bin': 203, 'max_cat_to_onehot': 2, 'max_cat_threshold': 4, 'n_estimators': 1828, 'sampling_method': 'gradient_based', 'subsample': 0.21905666352297487, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'enable_categorical': True}

Optuna Cross-Val AUC: 0.7021
Holdout Test AUC:     0.6959


### 100k

In [6]:
import optuna
import numpy as np
import warnings
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 30.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 1.0), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2, 10), # Class balancing weight for imbalanced target
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 5, 512), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "objective": "binary:logistic", # Binary classification objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
        
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0)
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBClassifier(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "scale_pos_weight",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_100k_parallel.html")


[I 2026-04-25 13:08:42,893] A new study created in memory with name: no-name-450990d6-8a6b-468c-8f1e-6dea458c7532
[I 2026-04-25 13:08:53,917] Trial 7 finished with value: 0.7037705148859494 and parameters: {'booster': 'gbtree', 'learning_rate': 0.13007085024957715, 'min_split_loss': 21.355755150673986, 'max_depth': 15, 'min_child_weight': 0.019758711224615796, 'max_delta_step': 5.448845201803208, 'colsample_bytree': 0.410101758400797, 'colsample_bylevel': 0.37732613638574936, 'colsample_bynode': 0.3549158448956422, 'reg_lambda': 1.0298053465023177e-05, 'reg_alpha': 3.5224340198839355, 'scale_pos_weight': 4.457022681690913, 'grow_policy': 'lossguide', 'max_leaves': 373, 'max_bin': 325, 'max_cat_to_onehot': 8, 'max_cat_threshold': 296, 'n_estimators': 147, 'sampling_method': 'uniform', 'subsample': 0.510279284221639}. Best is trial 7 with value: 0.7037705148859494.
[I 2026-04-25 13:09:08,124] Trial 5 finished with value: 0.6950207912777155 and parameters: {'booster': 'gbtree', 'learning_


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 14:32:54,268] Trial 317 finished with value: 0.7114291378144555 and parameters: {'booster': 'gbtree', 'learning_rate': 0.0097778531627673, 'min_split_loss': 4.842203730155134, 'max_depth': 14, 'min_child_weight': 4.8189392715177905e-05, 'max_delta_step': 5.576694427905714, 'colsample_bytree': 0.6439475102461473, 'colsample_bylevel': 0.4364026715507195, 'colsample_bynode': 0.4381446659797367, 'reg_lambda': 2.3679537437512099e-07, 'reg_alpha': 3.3531002020021714, 'scale_pos_weight': 2.9062909540419195, 'grow_policy': 'lossguide', 'max_leaves': 74, 'max_bin': 394, 'max_cat_to_onehot': 16, 'max_cat_threshold': 968, 'n_estimators': 888, 'sampling_method': 'uniform', 'subsample': 0.4037136324914682}. Best is trial 218 with value: 0.7133277282960735.
[I 2026-04-25 14:33:02,646] Trial 316 finished with value: 0.7119463068580677 and parameters: {'booster': 'gbtree', 'learning_rate': 0.00955719359527958, 'min_split_loss': 1.976059293867464, 'max_depth': 14, 'min_child_weight': 1.52


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 14:34:11,102] Trial 322 finished with value: 0.7114402545850531 and parameters: {'booster': 'gbtree', 'learning_rate': 0.008882579476637639, 'min_split_loss': 5.118875770023446, 'max_depth': 17, 'min_child_weight': 8.287365551782396e-05, 'max_delta_step': 5.646037761138231, 'colsample_bytree': 0.6416225038191417, 'colsample_bylevel': 0.6300869731202573, 'colsample_bynode': 0.431227041051254, 'reg_lambda': 4.085242448262244e-08, 'reg_alpha': 3.389672674142594, 'scale_pos_weight': 2.894896965851737, 'grow_policy': 'lossguide', 'max_leaves': 54, 'max_bin': 382, 'max_cat_to_onehot': 15, 'max_cat_threshold': 644, 'n_estimators': 888, 'sampling_method': 'uniform', 'subsample': 0.37373959372176163}. Best is trial 218 with value: 0.7133277282960735.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 14:34:38,499] Trial 319 finished with value: 0.7118104852082172 and parameters: {'booster': 'gbtree', 'learning_rate': 0.00895597131185323, 'min_split_loss': 5.676935321781597, 'max_depth': 14, 'min_child_weight': 4.865538474014669e-05, 'max_delta_step': 0.6252881229430756, 'colsample_bytree': 0.6239775337389726, 'colsample_bylevel': 0.6339233769027176, 'colsample_bynode': 0.42814486241435734, 'reg_lambda': 1.1287220964342843e-07, 'reg_alpha': 3.2942368513395133, 'scale_pos_weight': 2.358651077851495, 'grow_policy': 'lossguide', 'max_leaves': 491, 'max_bin': 414, 'max_cat_to_onehot': 14, 'max_cat_threshold': 641, 'n_estimators': 889, 'sampling_method': 'uniform', 'subsample': 0.37361353039170175}. Best is trial 218 with value: 0.7133277282960735.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 14:34:56,938] Trial 315 finished with value: 0.7094177461391167 and parameters: {'booster': 'gbtree', 'learning_rate': 0.009375568154489436, 'min_split_loss': 2.1352793683761218, 'max_depth': 14, 'min_child_weight': 4.6514761366054684e-05, 'max_delta_step': 5.529998550500919, 'colsample_bytree': 0.6342074026022737, 'colsample_bylevel': 0.6216890809551675, 'colsample_bynode': 0.44190616754312023, 'reg_lambda': 1.0998130497113899e-07, 'reg_alpha': 4.680229930263113, 'scale_pos_weight': 2.429418062388793, 'grow_policy': 'lossguide', 'max_leaves': 480, 'max_bin': 402, 'max_cat_to_onehot': 12, 'max_cat_threshold': 673, 'n_estimators': 880, 'sampling_method': 'uniform', 'subsample': 0.41118239313240595}. Best is trial 218 with value: 0.7133277282960735.
[I 2026-04-25 14:35:36,967] Trial 320 finished with value: 0.7086519726378465 and parameters: {'booster': 'gbtree', 'learning_rate': 0.009090495964111085, 'min_split_loss': 2.288210200894958, 'max_depth': 17, 'min_child_weight':


[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7133
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.009878389008570134,
    "min_split_loss": 1.6211471225466279,
    "max_depth": 18,
    "min_child_weight": 6.506529198811776,
    "max_delta_step": 4.996596322660088,
    "colsample_bytree": 0.5497212085853451,
    "colsample_bylevel": 0.6402797842177399,
    "colsample_bynode": 0.4945766078267192,
    "reg_lambda": 3.901817763099591e-08,
    "reg_alpha": 9.955075730252275,
    "scale_pos_weight": 2.385489120852272,
    "grow_policy": "lossguide",
    "max_leaves": 70,
    "max_bin": 428,
    "max_cat_to_onehot": 10,
    "max_cat_threshold": 686,
    "n_estimators": 901,
    "sampling_method": "uniform",
    "subsample": 0.34315380106756327,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.7038
  max_cat_threshold   : 0.0442
  n_estimators        : 0.0425
  colsample_bylevel   : 0.0299
  scale_pos_weight    : 0.02

In [7]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_xgboost.best_params.copy() 

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

if best_params["booster"] == "gbtree":
    # Apply trick only for gbtree
    best_params["n_estimators"] = original_trees * 10
    best_params["learning_rate"] = original_lr / 10
    print(f"\n[Scaling Trick Applied] GBTREE: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")
else:
    # Keep original parameters for DART
    print(f"\n[Scaling Trick Skipped] Booster is DART. Kept original values: Trees {original_trees}, LR {original_lr:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "binary:logistic"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_xgboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] GBTREE: Trees 901 -> 9010, LR 0.0099 -> 0.0010
BEST PARAMS: {'booster': 'gbtree', 'learning_rate': 0.0009878389008570133, 'min_split_loss': 1.6211471225466279, 'max_depth': 18, 'min_child_weight': 6.506529198811776, 'max_delta_step': 4.996596322660088, 'colsample_bytree': 0.5497212085853451, 'colsample_bylevel': 0.6402797842177399, 'colsample_bynode': 0.4945766078267192, 'reg_lambda': 3.901817763099591e-08, 'reg_alpha': 9.955075730252275, 'scale_pos_weight': 2.385489120852272, 'grow_policy': 'lossguide', 'max_leaves': 70, 'max_bin': 428, 'max_cat_to_onehot': 10, 'max_cat_threshold': 686, 'n_estimators': 9010, 'sampling_method': 'uniform', 'subsample': 0.34315380106756327, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'enable_categorical': True}

Optuna Cross-Val AUC: 0.7133
Holdout Test AUC:     0.7161


### Whole data set

In [16]:
import optuna
import numpy as np
import warnings
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

## Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 30.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 1.0), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2, 10), # Class balancing weight for imbalanced target
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 5, 512), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "objective": "binary:logistic", # Binary classification objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Row subsampling
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBClassifier(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "scale_pos_weight",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_full_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Classification/optuna_xgboost_full_parallel.html")


[I 2026-04-23 01:36:07,446] A new study created in memory with name: no-name-5c6f95d0-6cfe-44c2-b70b-282a0db82d38
[I 2026-04-23 01:37:11,992] Trial 5 finished with value: 0.7324143139106573 and parameters: {'booster': 'gbtree', 'learning_rate': 0.18137452556235562, 'min_split_loss': 17.54999685499812, 'max_depth': 2, 'min_child_weight': 0.00013387863982545998, 'max_delta_step': 2.809658325861837, 'colsample_bytree': 0.4674503211950386, 'colsample_bylevel': 0.7370066064162528, 'colsample_bynode': 0.3414181331064862, 'reg_lambda': 7.891849319663726e-07, 'reg_alpha': 2.209965110364385e-07, 'scale_pos_weight': 3.6102617970160846, 'grow_policy': 'lossguide', 'max_leaves': 297, 'max_bin': 388, 'max_cat_to_onehot': 47, 'max_cat_threshold': 517, 'n_estimators': 241, 'sampling_method': 'uniform', 'subsample': 0.27614963141980414}. Best is trial 5 with value: 0.7324143139106573.
[I 2026-04-23 01:38:26,515] Trial 8 finished with value: 0.7222519085125548 and parameters: {'booster': 'gbtree', 'lea


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 05:32:53,969] Trial 324 finished with value: 0.737562831117725 and parameters: {'booster': 'gbtree', 'learning_rate': 0.04327201222038523, 'min_split_loss': 8.422521471278863, 'max_depth': 7, 'min_child_weight': 0.005651951017541181, 'max_delta_step': 0.49643809823063256, 'colsample_bytree': 0.40574760123835835, 'colsample_bylevel': 0.5882065937461187, 'colsample_bynode': 0.9642954205487658, 'reg_lambda': 4.322583094834606, 'reg_alpha': 0.00039440307187170383, 'scale_pos_weight': 2.631333996939878, 'grow_policy': 'depthwise', 'max_leaves': 243, 'max_bin': 112, 'max_cat_to_onehot': 3, 'max_cat_threshold': 86, 'n_estimators': 961, 'sampling_method': 'uniform', 'subsample': 0.7853816370715303}. Best is trial 223 with value: 0.7387412082697721.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 05:33:27,165] Trial 325 finished with value: 0.7377302725538201 and parameters: {'booster': 'gbtree', 'learning_rate': 0.041273304125546535, 'min_split_loss': 8.507046331887807, 'max_depth': 7, 'min_child_weight': 0.014196214904362259, 'max_delta_step': 0.5385589227318301, 'colsample_bytree': 0.4204263474658033, 'colsample_bylevel': 0.6055544194640973, 'colsample_bynode': 0.9669112262880464, 'reg_lambda': 4.486262257140435, 'reg_alpha': 0.0002986358238738243, 'scale_pos_weight': 2.620155854163364, 'grow_policy': 'depthwise', 'max_leaves': 265, 'max_bin': 111, 'max_cat_to_onehot': 3, 'max_cat_threshold': 295, 'n_estimators': 966, 'sampling_method': 'uniform', 'subsample': 0.8076985933505443}. Best is trial 223 with value: 0.7387412082697721.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 05:36:03,871] Trial 330 finished with value: 0.7368034179628071 and parameters: {'booster': 'gbtree', 'learning_rate': 0.04354580315990979, 'min_split_loss': 7.418985614220479, 'max_depth': 8, 'min_child_weight': 0.01236068110068068, 'max_delta_step': 0.5590916377633903, 'colsample_bytree': 0.4168863028387652, 'colsample_bylevel': 0.5764314208103781, 'colsample_bynode': 0.9412358281339881, 'reg_lambda': 4.918447558987666, 'reg_alpha': 0.0003042492145241539, 'scale_pos_weight': 2.672738244086338, 'grow_policy': 'depthwise', 'max_leaves': 265, 'max_bin': 82, 'max_cat_to_onehot': 3, 'max_cat_threshold': 323, 'n_estimators': 929, 'sampling_method': 'uniform', 'subsample': 0.8058688268673002}. Best is trial 223 with value: 0.7387412082697721.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 05:36:10,970] Trial 328 finished with value: 0.7365191377612756 and parameters: {'booster': 'gbtree', 'learning_rate': 0.04087768874879841, 'min_split_loss': 7.515861864140987, 'max_depth': 9, 'min_child_weight': 0.0007919916452306916, 'max_delta_step': 0.5088671806347584, 'colsample_bytree': 0.42816403509782497, 'colsample_bylevel': 0.5899613246900481, 'colsample_bynode': 0.9424679671905184, 'reg_lambda': 4.910448174722238, 'reg_alpha': 0.0002814149971819137, 'scale_pos_weight': 2.6891083295263036, 'grow_policy': 'depthwise', 'max_leaves': 260, 'max_bin': 84, 'max_cat_to_onehot': 4, 'max_cat_threshold': 302, 'n_estimators': 967, 'sampling_method': 'uniform', 'subsample': 0.7936145064715644}. Best is trial 223 with value: 0.7387412082697721.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7387
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.03292776800197243,
    "min_split_loss": 8.210752775090127,
    "max_depth": 7,
    "min_child_weight": 0.15538873240656356,
    "max_delta_step": 0.27402281636668574,
    "colsample_bytree": 0.49786443299241995,
    "colsample_bylevel": 0.5536721341549702,
    "colsample_bynode": 0.6937873385849318,
    "reg_lambda": 5.867840852070052,
    "reg_alpha": 0.00036494609794361513,
    "scale_pos_weight": 2.197402352020041,
    "grow_policy": "depthwise",
    "max_leaves": 213,
    "max_bin": 81,
    "max_cat_to_onehot": 2,
    "max_cat_threshold": 31,
    "n_estimators": 967,
    "sampling_method": "uniform",
    "subsample": 0.693420186711674,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.5225
  scale_pos_weight    : 0.1313
  max_depth           : 0.1079
  n_estimators        : 0.0586
  colsample_bylevel   : 0.033

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_xgboost.best_params.copy() 

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

if best_params["booster"] == "gbtree":
    # Apply trick only for gbtree
    best_params["n_estimators"] = original_trees * 10
    best_params["learning_rate"] = original_lr / 10
    print(f"\n[Scaling Trick Applied] GBTREE: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")
else:
    # Keep original parameters for DART
    print(f"\n[Scaling Trick Skipped] Booster is DART. Kept original values: Trees {original_trees}, LR {original_lr:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "binary:logistic"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_xgboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] GBTREE: Trees 967 -> 9670, LR 0.0329 -> 0.0033
BEST_PARAMS: {'booster': 'gbtree', 'learning_rate': 0.0032927768001972434, 'min_split_loss': 8.210752775090127, 'max_depth': 7, 'min_child_weight': 0.15538873240656356, 'max_delta_step': 0.27402281636668574, 'colsample_bytree': 0.49786443299241995, 'colsample_bylevel': 0.5536721341549702, 'colsample_bynode': 0.6937873385849318, 'reg_lambda': 5.867840852070052, 'reg_alpha': 0.00036494609794361513, 'scale_pos_weight': 2.197402352020041, 'grow_policy': 'depthwise', 'max_leaves': 213, 'max_bin': 81, 'max_cat_to_onehot': 2, 'max_cat_threshold': 31, 'n_estimators': 9670, 'sampling_method': 'uniform', 'subsample': 0.693420186711674, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'eval_metric': 'auc', 'enable_categorical': True}

Optuna Cross-Val AUC: 0.7387
Holdout Test AUC:     0.7296


#### Check if 100k parameters hold on full dataset

Bad results, and optimal parametrs were found different so simplification can not be made.

In [18]:
# Check overfit on holdout set with best parameters from tuning

params_100k = {'booster': 'gbtree', 'learning_rate': 0.001223277058774911, 'min_split_loss': 3.606768308025944, 'max_depth': 11, 'min_child_weight': 0.016499320647867438, 'max_delta_step': 0.4200748985101004, 'colsample_bytree': 0.5756086532976907, 'colsample_bylevel': 0.4444718527495251, 'colsample_bynode': 0.23796818822800905, 'reg_lambda': 0.0007899079820553549, 'reg_alpha': 1.1988137677502194e-07, 'scale_pos_weight': 2.6973750500920275, 'grow_policy': 'lossguide', 'max_leaves': 49, 'max_bin': 86, 'max_cat_to_onehot': 30, 'max_cat_threshold': 362, 'n_estimators': 9340, 'sampling_method': 'uniform', 'subsample': 0.6199410549956432, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'enable_categorical': True}

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

final_model = XGBClassifier(**params_100k)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_xgboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


Optuna Cross-Val AUC: 0.7387
Holdout Test AUC:     0.7250


## Regression

[Parameters](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html)

### 1k

In [4]:
import optuna
import numpy as np
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 10.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree 
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 0.8), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "objective": "reg:squarederror", # Regression objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
        
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0)
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBRegressor(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_1k_parallel.html")


[I 2026-04-25 00:18:18,041] A new study created in memory with name: no-name-3f5ab2f2-4a62-4708-9a5e-5f623a4c3d87
[I 2026-04-25 00:18:18,559] Trial 7 finished with value: 0.3229718747888774 and parameters: {'booster': 'gbtree', 'learning_rate': 0.006669665235299899, 'min_split_loss': 2.851447591805055, 'max_depth': 5, 'min_child_weight': 0.3965750257674255, 'max_delta_step': 7.987935254751735, 'colsample_bytree': 0.5991284883019898, 'colsample_bylevel': 0.5544129452755414, 'colsample_bynode': 0.38465841802048995, 'reg_lambda': 3.8204135050666954, 'reg_alpha': 0.23244489415037922, 'grow_policy': 'lossguide', 'max_leaves': 5, 'max_bin': 104, 'max_cat_to_onehot': 15, 'max_cat_threshold': 5, 'n_estimators': 100, 'sampling_method': 'uniform', 'subsample': 0.7634104542103812}. Best is trial 7 with value: 0.3229718747888774.
[I 2026-04-25 00:18:19,278] Trial 0 finished with value: 0.3201881885477593 and parameters: {'booster': 'gbtree', 'learning_rate': 0.026532083223362955, 'min_split_loss':


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:19:18,076] Trial 218 finished with value: 0.32724315439530755 and parameters: {'booster': 'gbtree', 'learning_rate': 0.0434171382510925, 'min_split_loss': 0.3852953521990151, 'max_depth': 5, 'min_child_weight': 0.023187979184937955, 'max_delta_step': 0.830990982781586, 'colsample_bytree': 0.4701212021042281, 'colsample_bylevel': 0.3017019071517557, 'colsample_bynode': 0.745550967493309, 'reg_lambda': 0.007900854284378053, 'reg_alpha': 1.0236581271799214, 'grow_policy': 'depthwise', 'max_leaves': 21, 'max_bin': 80, 'max_cat_to_onehot': 14, 'max_cat_threshold': 12, 'n_estimators': 492, 'sampling_method': 'uniform', 'subsample': 0.26940988581249103}. Best is trial 114 with value: 0.316235573155014.
[I 2026-04-25 00:19:18,102] Trial 217 finished with value: 0.3185347564577353 and parameters: {'booster': 'gbtree', 'learning_rate': 0.0077251090432498845, 'min_split_loss': 0.39137805980796414, 'max_depth': 5, 'min_child_weight': 0.02471980382409185, 'max_delta_step': 0.416115


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:19:18,446] Trial 213 finished with value: 0.32235267803624396 and parameters: {'booster': 'gbtree', 'learning_rate': 0.05912674708319284, 'min_split_loss': 0.003348635888631435, 'max_depth': 5, 'min_child_weight': 0.025112647012191507, 'max_delta_step': 0.020281007490530634, 'colsample_bytree': 0.37152717164278626, 'colsample_bylevel': 0.343707452715937, 'colsample_bynode': 0.7308792261783762, 'reg_lambda': 0.020829122190721694, 'reg_alpha': 1.137704942362237, 'grow_policy': 'lossguide', 'max_leaves': 28, 'max_bin': 154, 'max_cat_to_onehot': 14, 'max_cat_threshold': 11, 'n_estimators': 796, 'sampling_method': 'uniform', 'subsample': 0.33640228832798}. Best is trial 114 with value: 0.316235573155014.
[I 2026-04-25 00:19:18,470] Trial 216 finished with value: 0.3201707315239608 and parameters: {'booster': 'gbtree', 'learning_rate': 0.011269292428132448, 'min_split_loss': 0.034779615129519104, 'max_depth': 5, 'min_child_weight': 0.0446470378198927, 'max_delta_step': 0.816


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.3162
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.006592924714393806,
    "min_split_loss": 0.3399903461254572,
    "max_depth": 7,
    "min_child_weight": 3.6857527823293634,
    "max_delta_step": 3.928223239451256,
    "colsample_bytree": 0.381061252694662,
    "colsample_bylevel": 0.3290330497906662,
    "colsample_bynode": 0.3640630952340392,
    "reg_lambda": 0.01062303202535693,
    "reg_alpha": 0.03812077743370024,
    "grow_policy": "lossguide",
    "max_leaves": 20,
    "max_bin": 169,
    "max_cat_to_onehot": 18,
    "max_cat_threshold": 11,
    "n_estimators": 537,
    "sampling_method": "uniform",
    "subsample": 0.5636865373558122,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.2964
  min_split_loss      : 0.2161
  colsample_bytree    : 0.1247
  max_bin             : 0.0904
  cols

In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_xgboost.best_params.copy()

best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "reg:squarederror"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_xgboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)

BEST_PARAMS: {'booster': 'gbtree', 'learning_rate': 0.006592924714393806, 'min_split_loss': 0.3399903461254572, 'max_depth': 7, 'min_child_weight': 3.6857527823293634, 'max_delta_step': 3.928223239451256, 'colsample_bytree': 0.381061252694662, 'colsample_bylevel': 0.3290330497906662, 'colsample_bynode': 0.3640630952340392, 'reg_lambda': 0.01062303202535693, 'reg_alpha': 0.03812077743370024, 'grow_policy': 'lossguide', 'max_leaves': 20, 'max_bin': 169, 'max_cat_to_onehot': 18, 'max_cat_threshold': 11, 'n_estimators': 537, 'sampling_method': 'uniform', 'subsample': 0.5636865373558122, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}

Optuna Cross-Val RMSE: 0.3162
Holdout Test RMSE:     0.3333


### 10k

In [6]:
import optuna
import numpy as np
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

## Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 10.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree 
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 0.8), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "objective": "reg:squarederror", # Regression objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
        
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0)
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBRegressor(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_10k_parallel.html")


[I 2026-04-25 00:19:22,164] A new study created in memory with name: no-name-66f95d76-bb35-4aee-83d0-4bb4d8a25563
[I 2026-04-25 00:19:23,595] Trial 6 finished with value: 0.3025583789560808 and parameters: {'booster': 'gbtree', 'learning_rate': 0.19184010539923876, 'min_split_loss': 3.1153220143384774, 'max_depth': 5, 'min_child_weight': 0.37206494529991596, 'max_delta_step': 9.277355064083755, 'colsample_bytree': 0.33422531153767765, 'colsample_bylevel': 0.5862927982908552, 'colsample_bynode': 0.7038299752117847, 'reg_lambda': 0.47191944997696605, 'reg_alpha': 0.024417557793676332, 'grow_policy': 'depthwise', 'max_leaves': 11, 'max_bin': 187, 'max_cat_to_onehot': 2, 'max_cat_threshold': 32, 'n_estimators': 150, 'sampling_method': 'gradient_based', 'subsample': 0.25389587370167194}. Best is trial 6 with value: 0.3025583789560808.
[I 2026-04-25 00:19:23,737] Trial 4 finished with value: 0.30058036809766686 and parameters: {'booster': 'gbtree', 'learning_rate': 0.08373352509472654, 'min_


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:24:12,040] Trial 347 finished with value: 0.29740215782125073 and parameters: {'booster': 'gbtree', 'learning_rate': 0.007664947249473291, 'min_split_loss': 1.220764771982464, 'max_depth': 4, 'min_child_weight': 0.05638755746858821, 'max_delta_step': 7.633379675169806, 'colsample_bytree': 0.7243246721762967, 'colsample_bylevel': 0.41193949232994537, 'colsample_bynode': 0.6355986935369592, 'reg_lambda': 2.1702908582917413, 'reg_alpha': 8.043241721386117, 'grow_policy': 'depthwise', 'max_leaves': 27, 'max_bin': 63, 'max_cat_to_onehot': 4, 'max_cat_threshold': 30, 'n_estimators': 958, 'sampling_method': 'gradient_based', 'subsample': 0.1474955356422285}. Best is trial 248 with value: 0.2964912605872014.
[I 2026-04-25 00:24:12,416] Trial 345 finished with value: 0.29822869111797934 and parameters: {'booster': 'gbtree', 'learning_rate': 0.0077659960067305075, 'min_split_loss': 1.619412356375883, 'max_depth': 5, 'min_child_weight': 0.05314390563121274, 'max_delta_step': 8.26


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:24:15,978] Trial 350 finished with value: 0.29686484433053345 and parameters: {'booster': 'gbtree', 'learning_rate': 0.006181139973839283, 'min_split_loss': 1.2053330298742915, 'max_depth': 4, 'min_child_weight': 0.05553458266691165, 'max_delta_step': 8.427602589984517, 'colsample_bytree': 0.6401779215028693, 'colsample_bylevel': 0.3086563858094084, 'colsample_bynode': 0.5291364667072855, 'reg_lambda': 2.3245517352268896, 'reg_alpha': 6.909609481496081, 'grow_policy': 'depthwise', 'max_leaves': 27, 'max_bin': 67, 'max_cat_to_onehot': 3, 'max_cat_threshold': 30, 'n_estimators': 965, 'sampling_method': 'gradient_based', 'subsample': 0.15142140502276424}. Best is trial 248 with value: 0.2964912605872014.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:24:16,434] Trial 352 finished with value: 0.29719995599384774 and parameters: {'booster': 'gbtree', 'learning_rate': 0.006697028392732719, 'min_split_loss': 1.6382355835982865, 'max_depth': 4, 'min_child_weight': 0.048044179057293024, 'max_delta_step': 7.435779492922476, 'colsample_bytree': 0.7257489159404473, 'colsample_bylevel': 0.30793365619230034, 'colsample_bynode': 0.6157789705746421, 'reg_lambda': 2.5586271873930873, 'reg_alpha': 6.7373978929114555, 'grow_policy': 'depthwise', 'max_leaves': 29, 'max_bin': 64, 'max_cat_to_onehot': 3, 'max_cat_threshold': 31, 'n_estimators': 958, 'sampling_method': 'gradient_based', 'subsample': 0.2534810244110057}. Best is trial 248 with value: 0.2964912605872014.
[I 2026-04-25 00:24:16,554] Trial 351 finished with value: 0.2973653631332157 and parameters: {'booster': 'gbtree', 'learning_rate': 0.006504824024341589, 'min_split_loss': 1.2001488604125148, 'max_depth': 4, 'min_child_weight': 0.05574281869805788, 'max_delta_step': 8.


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:24:16,854] Trial 353 finished with value: 0.29677391592501456 and parameters: {'booster': 'gbtree', 'learning_rate': 0.006662897572924038, 'min_split_loss': 1.1437299560802896, 'max_depth': 4, 'min_child_weight': 0.04553660987249841, 'max_delta_step': 0.5811038607786498, 'colsample_bytree': 0.7229420097746387, 'colsample_bylevel': 0.3093346128962459, 'colsample_bynode': 0.6120428432651432, 'reg_lambda': 0.015449202102741859, 'reg_alpha': 6.968076664611313, 'grow_policy': 'depthwise', 'max_leaves': 27, 'max_bin': 63, 'max_cat_to_onehot': 3, 'max_cat_threshold': 31, 'n_estimators': 985, 'sampling_method': 'gradient_based', 'subsample': 0.25896195604867817}. Best is trial 248 with value: 0.2964912605872014.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2965
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.0069298943309346744,
    "min_split_loss": 1.2225056392066438,
    "max_depth": 4,
    "min_child_weight": 0.044868703229338794,
    "max_delta_step": 7.322896593525407,
    "colsample_bytree": 0.664513006915934,
    "colsample_bylevel": 0.30183598693553104,
    "colsample_bynode": 0.6199112663715858,
    "reg_lambda": 0.022303464542099465,
    "reg_alpha": 8.45346616493749,
    "grow_policy": "depthwise",
    "max_leaves": 31,
    "max_bin": 69,
    "max_cat_to_onehot": 3,
    "max_cat_threshold": 32,
    "n_estimators": 995,
    "sampling_method": "gradient_based",
    "subsample": 0.19636351824285336,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4519
  max_bin             : 0.1656
  min_split_loss      : 0.0879
  max_delta_step      : 0.0641
  max_depth           : 0.0489
  max_leaves          : 0.0485
  c

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_xgboost.best_params.copy() 

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

if best_params["booster"] == "gbtree":
    # Apply trick only for gbtree
    best_params["n_estimators"] = original_trees * 2
    best_params["learning_rate"] = original_lr / 2
    print(f"\n[Scaling Trick Applied] GBTREE: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")
else:
    # Keep original parameters for DART
    print(f"\n[Scaling Trick Skipped] Booster is DART. Kept original values: Trees {original_trees}, LR {original_lr:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "reg:squarederror"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_xgboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GBTREE: Trees 995 -> 1990, LR 0.0069 -> 0.0035
BEST_PARAMS: {'booster': 'gbtree', 'learning_rate': 0.0034649471654673372, 'min_split_loss': 1.2225056392066438, 'max_depth': 4, 'min_child_weight': 0.044868703229338794, 'max_delta_step': 7.322896593525407, 'colsample_bytree': 0.664513006915934, 'colsample_bylevel': 0.30183598693553104, 'colsample_bynode': 0.6199112663715858, 'reg_lambda': 0.022303464542099465, 'reg_alpha': 8.45346616493749, 'grow_policy': 'depthwise', 'max_leaves': 31, 'max_bin': 69, 'max_cat_to_onehot': 3, 'max_cat_threshold': 32, 'n_estimators': 1990, 'sampling_method': 'gradient_based', 'subsample': 0.19636351824285336, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}

Optuna Cross-Val RMSE: 0.2965
Holdout Test RMSE:     0.3115


### 100k

In [8]:
import optuna
import numpy as np
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 30.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 1.0), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 5, 512), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "objective": "reg:squarederror", # Regression objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
        
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0)
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBRegressor(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_100k_parallel.html")


[I 2026-04-25 00:24:24,960] A new study created in memory with name: no-name-ca8427ca-a4b5-4b3b-a6ae-febae3e0fca2
[I 2026-04-25 00:24:35,015] Trial 5 finished with value: 0.300581278916075 and parameters: {'booster': 'gbtree', 'learning_rate': 0.06001779262860132, 'min_split_loss': 1.0210337624272214, 'max_depth': 8, 'min_child_weight': 4.878181313934955, 'max_delta_step': 3.915450790199577, 'colsample_bytree': 0.44690768732252634, 'colsample_bylevel': 0.7213305565391193, 'colsample_bynode': 0.6172177531885294, 'reg_lambda': 1.1150014262876837e-07, 'reg_alpha': 2.632706177108843e-06, 'grow_policy': 'depthwise', 'max_leaves': 437, 'max_bin': 280, 'max_cat_to_onehot': 2, 'max_cat_threshold': 17, 'n_estimators': 153, 'sampling_method': 'uniform', 'subsample': 0.7610647463447806}. Best is trial 5 with value: 0.300581278916075.
[I 2026-04-25 00:24:39,535] Trial 1 finished with value: 0.30590524942524056 and parameters: {'booster': 'gbtree', 'learning_rate': 0.006216089022347984, 'min_split_


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:39:37,013] Trial 226 finished with value: 0.3005765071512659 and parameters: {'booster': 'gbtree', 'learning_rate': 0.038044100874879284, 'min_split_loss': 1.38348610977361, 'max_depth': 10, 'min_child_weight': 0.061434093429648215, 'max_delta_step': 2.847302347566203, 'colsample_bytree': 0.3260553498019722, 'colsample_bylevel': 0.6887680183716893, 'colsample_bynode': 0.5079879444172025, 'reg_lambda': 1.915044724900809, 'reg_alpha': 0.2204007101127056, 'grow_policy': 'depthwise', 'max_leaves': 361, 'max_bin': 323, 'max_cat_to_onehot': 3, 'max_cat_threshold': 918, 'n_estimators': 983, 'sampling_method': 'uniform', 'subsample': 0.921938418678682}. Best is trial 126 with value: 0.29948885849576373.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:39:37,996] Trial 227 finished with value: 0.3007472795858515 and parameters: {'booster': 'gbtree', 'learning_rate': 0.05405480672110368, 'min_split_loss': 1.3852921174999917, 'max_depth': 10, 'min_child_weight': 0.059993084413021466, 'max_delta_step': 3.6229611474361776, 'colsample_bytree': 0.3182320672836059, 'colsample_bylevel': 0.7597012864279967, 'colsample_bynode': 0.4142930452860366, 'reg_lambda': 2.9074226797013563e-08, 'reg_alpha': 0.23925817243354822, 'grow_policy': 'depthwise', 'max_leaves': 312, 'max_bin': 324, 'max_cat_to_onehot': 19, 'max_cat_threshold': 911, 'n_estimators': 861, 'sampling_method': 'uniform', 'subsample': 0.9166827157138517}. Best is trial 126 with value: 0.29948885849576373.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:39:45,400] Trial 231 finished with value: 0.30078731541640125 and parameters: {'booster': 'gbtree', 'learning_rate': 0.05270984955531193, 'min_split_loss': 1.4222116666053028, 'max_depth': 5, 'min_child_weight': 0.24733190401444724, 'max_delta_step': 2.9167341523626633, 'colsample_bytree': 0.27081417836966226, 'colsample_bylevel': 0.6878137457791389, 'colsample_bynode': 0.4087782191246505, 'reg_lambda': 8.300930431777106, 'reg_alpha': 0.27518632141550303, 'grow_policy': 'depthwise', 'max_leaves': 307, 'max_bin': 311, 'max_cat_to_onehot': 48, 'max_cat_threshold': 437, 'n_estimators': 587, 'sampling_method': 'uniform', 'subsample': 0.9210705528378245}. Best is trial 126 with value: 0.29948885849576373.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 00:39:46,932] Trial 232 finished with value: 0.3007728920001827 and parameters: {'booster': 'gbtree', 'learning_rate': 0.05828656304462558, 'min_split_loss': 1.4539777764489572, 'max_depth': 10, 'min_child_weight': 0.04658207048096336, 'max_delta_step': 6.54567484684892, 'colsample_bytree': 0.31032913132857826, 'colsample_bylevel': 0.685399837309573, 'colsample_bynode': 0.4119214197299487, 'reg_lambda': 4.8096921420137315, 'reg_alpha': 0.22892866746023718, 'grow_policy': 'depthwise', 'max_leaves': 305, 'max_bin': 474, 'max_cat_to_onehot': 19, 'max_cat_threshold': 929, 'n_estimators': 608, 'sampling_method': 'uniform', 'subsample': 0.9202016407975278}. Best is trial 126 with value: 0.29948885849576373.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2995
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.0358874054592766,
    "min_split_loss": 0.30665873968801743,
    "max_depth": 9,
    "min_child_weight": 0.058208746224353215,
    "max_delta_step": 6.600824779885242,
    "colsample_bytree": 0.31452203665302875,
    "colsample_bylevel": 0.7845215630136946,
    "colsample_bynode": 0.45580656447386975,
    "reg_lambda": 1.4591079946865122,
    "reg_alpha": 7.300918108293281,
    "grow_policy": "depthwise",
    "max_leaves": 311,
    "max_bin": 270,
    "max_cat_to_onehot": 25,
    "max_cat_threshold": 578,
    "n_estimators": 924,
    "sampling_method": "uniform",
    "subsample": 0.892493539712304,
}

--- PARAMETER IMPORTANCE ---
  min_split_loss      : 0.6541
  n_estimators        : 0.0933
  max_cat_threshold   : 0.0676
  learning_rate       : 0.0525
  max_cat_to_onehot   : 0.0380
  colsample_bynode    : 0.0214
  colsamp

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_xgboost.best_params.copy() 

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

if best_params["booster"] == "gbtree":
    # Apply trick only for gbtree
    best_params["n_estimators"] = original_trees * 10
    best_params["learning_rate"] = original_lr / 10
    print(f"\n[Scaling Trick Applied] GBTREE: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")
else:
    # Keep original parameters for DART
    print(f"\n[Scaling Trick Skipped] Booster is DART. Kept original values: Trees {original_trees}, LR {original_lr:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "reg:squarederror"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_xgboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GBTREE: Trees 924 -> 9240, LR 0.0359 -> 0.0036
BEST_PARAMS: {'booster': 'gbtree', 'learning_rate': 0.00358874054592766, 'min_split_loss': 0.30665873968801743, 'max_depth': 9, 'min_child_weight': 0.058208746224353215, 'max_delta_step': 6.600824779885242, 'colsample_bytree': 0.31452203665302875, 'colsample_bylevel': 0.7845215630136946, 'colsample_bynode': 0.45580656447386975, 'reg_lambda': 1.4591079946865122, 'reg_alpha': 7.300918108293281, 'grow_policy': 'depthwise', 'max_leaves': 311, 'max_bin': 270, 'max_cat_to_onehot': 25, 'max_cat_threshold': 578, 'n_estimators': 9240, 'sampling_method': 'uniform', 'subsample': 0.892493539712304, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}

Optuna Cross-Val RMSE: 0.2995
Holdout Test RMSE:     0.2983


### Whole data set

In [10]:
import optuna
import numpy as np
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

## Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_xgboost(trial):
    params = {
        "booster": trial.suggest_categorical("booster", ["gbtree"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "min_split_loss": trial.suggest_float("min_split_loss", 0, 30.0), # Min loss reduction required to make a split
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 10.0), # Max delta step we allow each tree's weight estimation to be
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 1.0), # % of features used per level
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True), # L2 regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True), # L1 regularization
        "tree_method": "hist", # Use histogram-based algorithm
        "grow_policy": trial.suggest_categorical("grow_policy", ["lossguide", "depthwise"]), # Tree growth policy
        "max_leaves": trial.suggest_int("max_leaves", 5, 512), # Max leaves per tree. Low to prevent memorizing small data
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for continuous features
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding of categorical features
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "objective": "reg:squarederror", # Regression objective
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbosity": 0, # Suppress XGBoost output
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }

    if params["booster"] == "dart": # Dropouts meet Multiple Additive Regression Trees (DART)
        params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"]) # Type of sampling algorithm for dropout
        params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"]) # Type of normalization algorithm for dropout
        params["rate_drop"] = trial.suggest_float("rate_drop", 0.1, 0.5) # Dropout rate
        params["one_drop"] = trial.suggest_categorical("one_drop", [0, 1]) # Whether to drop at least one tree in each dropout iteration
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.1, 0.5) # Probability of skipping dropout entirely
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Row subsampling
    else:
        params["sampling_method"] = trial.suggest_categorical("sampling_method", ["uniform", "gradient_based"]) # Sampling method for GBDT. Uniform is standard row subsampling, gradient_based is GOSS-like sampling based on gradient magnitudes
        if params["sampling_method"] == "gradient_based":
            params["subsample"] = trial.suggest_float("subsample", 0.05, 1.0) # Row subsampling
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Row subsampling
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = XGBRegressor(**params, enable_categorical=True)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr,
                verbose=False
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_xgboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgboost.optimize(objective_xgboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_xgboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_xgboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_xgboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_xgboost)
fig1.show()
    
fig2 = plot_param_importances(study_xgboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_xgboost, 
    params=[
        "booster",
        "learning_rate",
        "min_split_loss",
        "max_depth",
        "min_child_weight",
        "max_delta_step",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "reg_lambda",
        "reg_alpha",
        "grow_policy",
        "max_leaves",
        "max_bin",
        "max_cat_to_onehot",
        "max_cat_threshold",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_full_history.html")
fig2.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/XGBoost/Regression/optuna_xgboost_full_parallel.html")


[I 2026-04-25 00:40:30,235] A new study created in memory with name: no-name-3dfedcd3-2b63-4d6e-9bc2-72c6e09d3bc6
[I 2026-04-25 00:41:20,452] Trial 4 finished with value: 0.23516838722546773 and parameters: {'booster': 'gbtree', 'learning_rate': 0.06973202499595622, 'min_split_loss': 7.490513945101311, 'max_depth': 8, 'min_child_weight': 1.4113654920187015, 'max_delta_step': 8.359165361567412, 'colsample_bytree': 0.8404852912279446, 'colsample_bylevel': 0.2752720625272634, 'colsample_bynode': 0.9489014212691862, 'reg_lambda': 0.0006306615112466985, 'reg_alpha': 5.377139539770213e-05, 'grow_policy': 'depthwise', 'max_leaves': 425, 'max_bin': 253, 'max_cat_to_onehot': 10, 'max_cat_threshold': 264, 'n_estimators': 147, 'sampling_method': 'gradient_based', 'subsample': 0.3432445005791084}. Best is trial 4 with value: 0.23516838722546773.
[I 2026-04-25 00:41:38,118] Trial 7 finished with value: 0.23464132526548398 and parameters: {'booster': 'gbtree', 'learning_rate': 0.0956360638092186, 'm


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 03:36:24,835] Trial 288 finished with value: 0.2335464246790968 and parameters: {'booster': 'gbtree', 'learning_rate': 0.01283249982835296, 'min_split_loss': 0.9371231396881785, 'max_depth': 14, 'min_child_weight': 0.0007780789493934134, 'max_delta_step': 1.510358569238782, 'colsample_bytree': 0.7277977196831789, 'colsample_bylevel': 0.4386982230021067, 'colsample_bynode': 0.7790494210611654, 'reg_lambda': 3.131981490863506e-08, 'reg_alpha': 0.0057001361882242155, 'grow_policy': 'lossguide', 'max_leaves': 291, 'max_bin': 381, 'max_cat_to_onehot': 21, 'max_cat_threshold': 636, 'n_estimators': 636, 'sampling_method': 'uniform', 'subsample': 0.9426736647629659}. Best is trial 189 with value: 0.23289783359820623.
[I 2026-04-25 03:36:28,996] Trial 293 finished with value: 0.23405146634441767 and parameters: {'booster': 'gbtree', 'learning_rate': 0.15371164300707657, 'min_split_loss': 1.0116716854370533, 'max_depth': 14, 'min_child_weight': 1.5655290316656918, 'max_delta_step':


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 03:37:19,392] Trial 284 finished with value: 0.23293542434196576 and parameters: {'booster': 'gbtree', 'learning_rate': 0.01285057949220331, 'min_split_loss': 0.0678152821767789, 'max_depth': 13, 'min_child_weight': 3.154556898030383, 'max_delta_step': 3.592231025738443, 'colsample_bytree': 0.8007461729133648, 'colsample_bylevel': 0.35177159492699767, 'colsample_bynode': 0.7761689207247193, 'reg_lambda': 1.3818517900959166e-06, 'reg_alpha': 0.0051434215508083065, 'grow_policy': 'lossguide', 'max_leaves': 357, 'max_bin': 328, 'max_cat_to_onehot': 15, 'max_cat_threshold': 728, 'n_estimators': 718, 'sampling_method': 'uniform', 'subsample': 0.9364294668682662}. Best is trial 189 with value: 0.23289783359820623.
[I 2026-04-25 03:38:08,783] Trial 290 finished with value: 0.23358259820958124 and parameters: {'booster': 'gbtree', 'learning_rate': 0.010525452374067561, 'min_split_loss': 1.0437596813444092, 'max_depth': 14, 'min_child_weight': 3.381558202622042, 'max_delta_step': 


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 03:38:17,770] Trial 287 finished with value: 0.23301971781552655 and parameters: {'booster': 'gbtree', 'learning_rate': 0.012887063250040603, 'min_split_loss': 0.01988924739845313, 'max_depth': 13, 'min_child_weight': 2.9829494909607295, 'max_delta_step': 1.4864464157988997, 'colsample_bytree': 0.7300275188210927, 'colsample_bylevel': 0.3450915543752462, 'colsample_bynode': 0.7130397691665574, 'reg_lambda': 0.00298107652378736, 'reg_alpha': 0.0058853130967677116, 'grow_policy': 'lossguide', 'max_leaves': 282, 'max_bin': 381, 'max_cat_to_onehot': 21, 'max_cat_threshold': 639, 'n_estimators': 619, 'sampling_method': 'uniform', 'subsample': 0.9401938740266726}. Best is trial 189 with value: 0.23289783359820623.
[I 2026-04-25 03:38:30,920] Trial 292 finished with value: 0.23358974311271616 and parameters: {'booster': 'gbtree', 'learning_rate': 0.010469576631229703, 'min_split_loss': 1.0300679002665858, 'max_depth': 14, 'min_child_weight': 3.6651857789205557, 'max_delta_step':


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 03:39:21,184] Trial 291 finished with value: 0.2330226366035745 and parameters: {'booster': 'gbtree', 'learning_rate': 0.01291100745510275, 'min_split_loss': 0.013043411490342965, 'max_depth': 14, 'min_child_weight': 3.641419517199321, 'max_delta_step': 4.713438181313291, 'colsample_bytree': 0.7344437422705964, 'colsample_bylevel': 0.46365632643778193, 'colsample_bynode': 0.7163891635460016, 'reg_lambda': 6.814202604218474e-06, 'reg_alpha': 0.0026146897184284113, 'grow_policy': 'lossguide', 'max_leaves': 259, 'max_bin': 386, 'max_cat_to_onehot': 21, 'max_cat_threshold': 656, 'n_estimators': 621, 'sampling_method': 'uniform', 'subsample': 0.9455396520793006}. Best is trial 189 with value: 0.23289783359820623.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2329
BEST PARAMETERS:
best_params = {
    "booster": "gbtree",
    "learning_rate": 0.015361738364973563,
    "min_split_loss": 0.019158435834981884,
    "max_depth": 12,
    "min_child_weight": 5.46291036872166,
    "max_delta_step": 4.8592833061852465,
    "colsample_bytree": 0.78840555707391,
    "colsample_bylevel": 0.3400841955942318,
    "colsample_bynode": 0.7797771602511868,
    "reg_lambda": 0.001536038127856472,
    "reg_alpha": 0.00444179665468952,
    "grow_policy": "lossguide",
    "max_leaves": 303,
    "max_bin": 352,
    "max_cat_to_onehot": 18,
    "max_cat_threshold": 668,
    "n_estimators": 680,
    "sampling_method": "uniform",
    "subsample": 0.9347372849225181,
}

--- PARAMETER IMPORTANCE ---
  min_split_loss      : 0.7451
  min_child_weight    : 0.0873
  max_leaves          : 0.0530
  max_cat_threshold   : 0.0363
  max_bin             : 0.0234
  max_delta_step      : 0.0091
  n_est

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_xgboost.best_params.copy() 

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

if best_params["booster"] == "gbtree":
    # Apply trick only for gbtree
    best_params["n_estimators"] = original_trees * 10
    best_params["learning_rate"] = original_lr / 10
    print(f"\n[Scaling Trick Applied] GBTREE: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")
else:
    # Keep original parameters for DART
    print(f"\n[Scaling Trick Skipped] Booster is DART. Kept original values: Trees {original_trees}, LR {original_lr:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = 0
best_params["tree_method"] = "hist"
best_params["objective"] = "reg:squarederror"
best_params["enable_categorical"] = True

print(f"BEST PARAMS: {best_params}")

final_model = XGBRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_xgboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GBTREE: Trees 680 -> 6800, LR 0.0154 -> 0.0015
BEST_PARAMS: {'booster': 'gbtree', 'learning_rate': 0.0015361738364973562, 'min_split_loss': 0.019158435834981884, 'max_depth': 12, 'min_child_weight': 5.46291036872166, 'max_delta_step': 4.8592833061852465, 'colsample_bytree': 0.78840555707391, 'colsample_bylevel': 0.3400841955942318, 'colsample_bynode': 0.7797771602511868, 'reg_lambda': 0.001536038127856472, 'reg_alpha': 0.00444179665468952, 'grow_policy': 'lossguide', 'max_leaves': 303, 'max_bin': 352, 'max_cat_to_onehot': 18, 'max_cat_threshold': 668, 'n_estimators': 6800, 'sampling_method': 'uniform', 'subsample': 0.9347372849225181, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}

Optuna Cross-Val RMSE: 0.2329
Holdout Test RMSE:     0.2835
